# InternVL3 P2.1 shard0: one offline Run All

Owner-only after all audit/merge/authority gates. Kaggle Settings: **Internet OFF**, **Tesla T4** (child observes exactly one GPU). Attach new confirmed-source prep output; existing **InternVL3 Source + Model Prep FIX v2** and **InternVL3 Runtime Prep FIX v2** outputs; original InspecSafe dataset, exact source manifest/provenance; and historical `internvl3-p2-shard0-first` result package or its actual manifest and status. Runtime must include exact managed Python 3.11.11 and unchanged pinned site-packages; no model download/runtime rebuild is performed. Fill release SHA and attachment controls, then run **Run All once**. Refuse reruns/resumes. See `notes/w2_internvl3_p21_kaggle_runbook.md`.

In [ ]:
# Fill only the verified release SHA and owner attachment controls.
# Empty attachment paths request exact-content discovery. If ambiguous, provide
# paths relative to Kaggle's attached input root (dataset-slug/subdir/file).
SETTINGS = {
    "final_merge_sha": "",
    "kaggle_internet_off_confirmed": False,
    "source_release_manifest": "",
    "source_model_input": "",
    "runtime_input": "",
    "runtime_site_packages": "",
    "historical_results_input": "",
    "dataset_root": "",
    "manifest_path": "",
    "provenance_path": "",
}


In [ ]:
# Single Run All runs only internvl3-p21-shard0-rerun1. This cell never retries.
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import tarfile

work = Path.cwd()
inputs = work.parent / "input"
output = work / "internvl3_p21_shard0_output"
repo = None
helper_loaded = False
try:
    sha = SETTINGS["final_merge_sha"]
    if len(sha) != 40 or any(c not in "0123456789abcdef" for c in sha):
        raise RuntimeError("Exact post-merge verified release SHA required")
    if SETTINGS["kaggle_internet_off_confirmed"] is not True:
        raise RuntimeError("Set Kaggle Settings Internet OFF, then confirm in SETTINGS")
    if SETTINGS["source_release_manifest"]:
        relative = PurePosixPath(SETTINGS["source_release_manifest"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Source release manifest path must be input-relative")
        manifests = [inputs / relative]
    else:
        manifests = [p for p in inputs.rglob("p21_source_release.json")
                     if json.loads(p.read_text())["final_merge_sha"] == sha]
    if len(manifests) != 1:
        raise RuntimeError("Attach/select exact new post-merge source prep output")
    metadata = json.loads(manifests[0].read_text())
    if (metadata["final_merge_sha"] != sha or metadata["archive"] != "internvl3_p21_source.tar.gz"
            or metadata["base_sha"] != "f0b5ea775b3c5bbe5e8618ba1372e59b8b8e150f"):
        raise RuntimeError("Exact P2.1 source package metadata required")
    archive = manifests[0].parent / metadata["archive"]
    digest = hashlib.sha256()
    with archive.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != metadata["archive_sha256"]:
        raise RuntimeError("Source archive SHA-256 mismatch")
    destination = work / "p21_release"
    destination.mkdir(exist_ok=False)
    # Source packages are regular files only. Reject traversal/links/devices
    # before extracting, independently of code inside the source package.
    with tarfile.open(archive, "r:gz") as stream:
        members = stream.getmembers()
        names = set()
        for member in members:
            path = PurePosixPath(member.name)
            if (path.is_absolute() or ".." in path.parts or "\\" in member.name
                    or ":" in member.name or member.name in names
                    or not (member.isfile() or member.isdir())):
                raise RuntimeError("Unsafe source package member")
            names.add(member.name)
        stream.extractall(destination, filter="data")
    repo = destination / "SafeShift"
    def git(*args):
        return subprocess.check_output(["git", "-C", str(repo), *args]).decode().strip()
    if git("rev-parse", "HEAD") != sha:
        raise RuntimeError("Offline source HEAD mismatch")
    parents = git("rev-list", "--parents", "-n", "1", "HEAD").split()
    if len(parents) != 3 or parents[1] != "f0b5ea775b3c5bbe5e8618ba1372e59b8b8e150f":
        raise RuntimeError("Offline Standard Merge BASE parent required")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("Dirty offline source")
    sys.path.insert(0, str(repo))
    from safeshift.runners.p21_kaggle import run_offline
    helper_loaded = True
    summary = run_offline(repo, inputs, SETTINGS, output)
    print("STOP_BEFORE_SHARDS_1_3=" + str(summary["stop_before_shards_1_3"]))
except BaseException as error:
    # Bootstrap failures also produce an owner package, even if source code
    # could not be imported. Never manufacture a harness run_manifest/status.
    if output.exists():
        print("Existing output preserved; refusing overwrite/retry: " + str(output))
        raise
    output.mkdir(exist_ok=False)
    summary = {
        "run_id": "internvl3-p21-shard0-rerun1", "protocol_version": "P2.1",
        "execution_status": "BOOTSTRAP_FAILED_NO_PRODUCTION_CALL",
        "SUCCESS": 0, "INVALID": 0, "FAILED": 0, "NOT_ATTEMPTED": 1254,
        "zero_canonical_yield": True, "stop_before_shards_1_3": True,
        "failure": {"type": type(error).__name__, "message": str(error)},
    }
    summary_path = output / "owner_result_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    package = output / "internvl3-p21-shard0-rerun1_results.tar.gz"
    with tarfile.open(package, "x:gz") as stream:
        stream.add(summary_path, arcname=summary_path.name)
    print(json.dumps(summary, indent=2))
    print("RESULT_PACKAGE=" + str(package))


Save `internvl3_p21_shard0_output` after COMPLETED, FAILED or 100% INVALID. Summary prints SUCCESS, INVALID, FAILED, NOT_ATTEMPTED and zero_canonical_yield. If zero canonical yield or generation/runtime failure, preserve output and stop before shards1-3. This notebook executes only shard0. P2.1 acceptance was proposed after seeing InspecSafe outputs; disclose benchmark adaptation risk and keep historical P2 evidence separate. Existing Qwen2.5 P2 is not P2.1; Qwen3/Moondream remain unauthorized. Kernel/VM loss cannot be caught by Python; partial results must still be saved and never resumed.